In [1]:
# =============================================================================
# BASE IMPORTS, LOGGING SETUP
# =============================================================================
from trading.backtester.utils       import setup_logging
from trading.backtester.engine      import BacktestRunner, load_config, RunContext
from trading.backtester.performance import PerformanceReport

run = RunContext.create()
run_dir = run.run_dir
setup_logging(run_dir=run_dir)

backtest_config = load_config(file = 'default_backtest.yaml')



In [2]:
# =============================================================================
# STRATEGY CONSTRUCTION
# Replace ONLY this section when testing a different strategy
# =============================================================================

# 1. Import strategy + config
from trading.strategy_engine.strategies.directional import (
    MACrossover,
    MACrossoverConfig,
)

from trading.strategy_engine.features.trend import MAType


# 2. Define strategy parameters
strategy_config = MACrossoverConfig(
    fast_period=20,
    fast_type=MAType.EMA,
    slow_period=50,
    slow_type=MAType.SMA,
)


# 3. Build strategy object
strategy = MACrossover(strategy_config)


In [3]:
# =============================================================================
# RUN STRATEGY
# =============================================================================

run.save_configs(backtest_config, strategy_config)  

runner  = BacktestRunner(backtest_config, strategy)

try:
    results = runner.run()
    (run_dir / "SUCCESS").touch()

except Exception:
    (run_dir / "FAILED").touch()
    raise


results.save(run_dir)

2026-07-20 11:11:04 | INFO     | trading.backtester.engine.runner | Starting backtest run: MA_CROSS_ema_20_sma_50
2026-07-20 11:11:04 | INFO     | trading.backtester.engine.runner | Loaded 8760 bars from 2025-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
2026-07-20 11:11:04 | INFO     | trading.backtester.engine.runner | Validated 8760 signals — no lookahead detected.
2026-07-20 11:11:04 | INFO     | trading.backtester.engine.runner | Phase 1 complete — 8760 bars, 215 signals.
2026-07-20 11:11:05 | INFO     | trading.backtester.engine.runner | Backtest complete: 8760 bars processed, final equity=30147.50
2026-07-20 11:11:05 | INFO     | trading.backtester.engine.results | Successfully saved backtest results to C:\Users\SBBus\PROJECTS\trading\runs\20260720_01_963


In [ ]:
print(results.trade_log.closed_trades)
print(results.portfolio_history.tail(25))

In [ ]:
# =============================================================================
# RESULTS
# =============================================================================


report = PerformanceReport(results.portfolio_history)
### Save report here
#report.save(run_dir)

report.display_report("BTCUSDT")
